# Instruccions

L'exàmen consta d'una pregunta (2/10 punts) i dos exercicis (4/10 punts cada un). 

Per a poder fer els exercicis heu de descarregar el fitxer `dataset.csv` del campus virtual i el fitxer `practica3.py`.

In [1]:
import numpy as np
import pandas as pd

import practica3 as pt3

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import practica3 as pt3

# Cargar datos
df = pd.read_csv('dataset.csv')
texts = df['Text'].tolist()
labels = df['language'].tolist()

# train/test
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)
# Extraer n-gramas 
X_train_ngrams = pt3.extract_ngrams_from_texts(X_train, n=2)
X_test_ngrams = pt3.extract_ngrams_from_texts(X_test, n=2)

## Pregunta

+ Perquè els bigrames oberts són mes robusts davant de faltes d'ortografies o errors tipogràfics respecte els bigrames tancats?
+ Il·lustra-ho usant l'exemple de la paraula `HELLO`.

In [4]:
# la teva resposta aquí:

# Los bigramas abiertos son más robustos porque no exigen que las letras estén pegadas una al lado de la otra. 
# Esto hace que aguanten mejor los errores tipográficos. Si te comes una letra o pones una de más, los bigramas normales (cerrados) se rompen justo ahí y pierdes la pista.
# Los abiertos, al dar saltos, siguen capturando la estructura general de la palabra ignorando el error local.

# Ejemplo con HELLO vs HELO (nos dejamos una L):
# - Los cerrados fallarían al buscar la "LL" exacta.
# - Los abiertos siguen funcionando porque conectan letras distantes, como la "H" del principio con la "O" del final, permitiendo identificar la palabra aunque esté mal escrita.

## Exercici 1

Amb el TfidVectorizer entrenat de la pràctica, agafa la primera frase de `X_test` i troba el bigrama amb TF-IDF més alt.

> Pista: ajuda't de la funció de sklearn `vectorizer.get_feature_names_out()` (retorna la llista de característiques que el vectoritzador ha generat després d’ajustar-se al text).





In [7]:
# la teva resposta aquí
# vectorizar
vectorizer = TfidfVectorizer(analyzer=lambda x: x)
vectorizer.fit(X_train_ngrams)
X_test_tfidf = vectorizer.transform(X_test_ngrams)

# bigramas
feature_names = vectorizer.get_feature_names_out()

# vector disperso primera frase de X_test
first_sentence_vector = X_test_tfidf[0]

# indice donde valor es mayor
max_idx = first_sentence_vector.argmax()

# bigrama y su puntuacion
top_bigram = feature_names[max_idx]
top_score = first_sentence_vector[0, max_idx]

print(f"Bigrama con mayor TF-IDF: '{top_bigram}'")
print(f"Valor: {top_score}")


Bigrama con mayor TF-IDF: '年_'
Valor: 0.08689305080728892


## Exercici 2

Implementa la funció `get_top_features(idioma, k=10)` que retorna els $k$ bigrames més freqüents de `X_train_ngrams` de l'idioma donat, la seva *log-probability* i el nombre de vegades que apareixen.

> La implementació ha de ser **pura en python** sense utilitzar cap funció de scikit-learn.
> Fes servir `Counter()` per comptar les aparicions de cada bigrama, i `Counter.most_common()` per obtenir els $k$ bigrams més freqüents.



```python
get_top_features('Spanish', 10)

hauria de retornar / imprimir:

e_: -7.1610 (Count: 801)
en: -7.1647 (Count: 798)
a_: -7.1697 (Count: 794)
de: -7.1735 (Count: 791)
n_: -7.1735 (Count: 791)
ra: -7.1735 (Count: 791)
ea: -7.1735 (Count: 791)
es: -7.1735 (Count: 791)
er: -7.1735 (Count: 791)
co: -7.1773 (Count: 788)

In [18]:
import math
from collections import Counter

def get_top_features_pure_python(language, k=10):
    """
    Obtenir el top k features per una llengua donada fent servir només python.
    
    Args:
        language (str): la classe o llengua objectiu
        k (int): nombre de features a obtenir
    """
    # Calcular el vocabulario total (V)
    all_ngrams = set()
    for ngrams in X_train_ngrams:
        all_ngrams.update(ngrams)
    V = len(all_ngrams)
    
    # Contar
    lang_ngrams = []
    for text_ngrams, label in zip(X_train_ngrams, y_train):
        if label == language:
            unique_ngrams = set(text_ngrams)
            lang_ngrams.extend(unique_ngrams)
            
    # Contar frecuencias
    counts = Counter(lang_ngrams)
    
    # k más frecuentes
    top_features = counts.most_common(k)
    
    # Log Prob
    total_count_class = sum(counts.values())
    denominator = total_count_class + V
    
    results = []
    for feature, count in top_features:
        log_prob = math.log((count + 1) / denominator)
        print(f"{feature}: {log_prob:.4f} (Count: {count})")
        results.append((feature, log_prob, count))
        
    return results

In [20]:
# Test de Español Jose   (Da orden diferente pero son los mismos bigramas, los desordenados tienen el mismo número de counts :) )

result = get_top_features_pure_python('Spanish', 10)

e_: -7.1610 (Count: 801)
en: -7.1647 (Count: 798)
a_: -7.1697 (Count: 794)
es: -7.1735 (Count: 791)
er: -7.1735 (Count: 791)
ea: -7.1735 (Count: 791)
ra: -7.1735 (Count: 791)
n_: -7.1735 (Count: 791)
de: -7.1735 (Count: 791)
_d: -7.1773 (Count: 788)


In [21]:
#test 

result = get_top_features_pure_python('Chinese', 10)

_年: -8.3592 (Count: 415)
在的: -8.4550 (Count: 377)
的的: -8.5011 (Count: 360)
是的: -8.5320 (Count: 349)
_在: -8.6216 (Count: 319)
年月: -8.7235 (Count: 288)
一的: -8.7623 (Count: 277)
了的: -8.8448 (Count: 255)
月日: -8.8448 (Count: 255)
有的: -8.8685 (Count: 249)
